# 02 · Data Modeling
**Goal:** Decide how the four tables relate to each other, validate that the join keys actually work, and design the schema of the master analytical dataset — before writing any cleaning or merge code.

## Load the four raw datasets
Same four tables as the data_audit notebook. However, this notebook checks the relationships *between* them rather than each table on its own.

In [3]:
import pandas as pd

menu         = pd.read_csv('../data/raw/starbucks_menu.csv')
transactions = pd.read_csv('../data/raw/synthetic_transactions.csv')
macro        = pd.read_csv('../data/raw/fred_macro.csv')
weather      = pd.read_csv('../data/raw/weather_daily.csv')

## Star schema

`Transaction` is by far the largest and most granular table — every other table describes something a transaction references (what was bought, what the weather was, what the economy looked like that month). That makes `Transaction` the **fact table**, and `Menu`, `Weather`, `Macro` the **dimension tables**.

In [4]:
for name, df in [('menu', menu), ('transactions', transactions), ('macro', macro), ('weather', weather)]:
    print(f'{name:15s} {df.shape[0]:>7,} rows  {df.shape[1]:>2} cols')

menu                242 rows   8 cols
transactions    100,000 rows  20 cols
macro                60 rows   4 cols
weather           3,655 rows   5 cols


## Validating the join keys

A diagram is a hypothesis. Before trusting it, check that every relationship actually holds in the raw data.

**`(product_name, size)`: Menu vs. Transaction**

In [5]:
def keyset(df, cols):
    return set(df[cols].drop_duplicates().itertuples(index=False, name=None))

menu_keys = keyset(menu, ['product_name', 'size'])
tx_keys   = keyset(transactions, ['product_name', 'size'])

print('menu unique (product_name, size):       ', len(menu_keys))
print('transaction unique (product_name, size):', len(tx_keys))
print('transaction keys missing from menu:     ', tx_keys - menu_keys)

menu unique (product_name, size):        106
transaction unique (product_name, size): 106
transaction keys missing from menu:      set()


**`category` and `city`**

In [6]:
print('category values in transactions but not menu:', set(transactions['category']) - set(menu['category']))
print('city values in transactions but not weather:  ', set(transactions['city']) - set(weather['city']))

category values in transactions but not menu: set()
city values in transactions but not weather:   set()


Both join keys check out cleanly — every `(product_name, size)` combination, every `category`, and every `city` in Transaction has a match on the dimension side. That confirms the `WEATHER — TRANSACTION` and category/city relationships are safe to build on.

## The problem: Menu is not one row per product

106 unique `(product_name, size)` combinations, but how many actual rows does Menu have?

In [7]:
print('menu shape:', menu.shape)
print('unique (product_name, size) combinations:', len(menu_keys))

menu shape: (242, 8)
unique (product_name, size) combinations: 106


242 rows for 106 keys means most products appear more than once. That alone could just be exact duplicate rows — harmless. The real question is whether the *values* differ between rows that share a key, because if they do, a plain JOIN on `(product_name, size)` will fan out: one transaction row will match several menu rows and multiply itself in the merge.

In [8]:
dup_groups = menu.groupby(['product_name', 'size'])
n_varying = sum(
    1 for _, g in dup_groups
    if len(g) > 1 and g.drop(columns=['product_name', 'size']).nunique().gt(1).any()
)
print(f'{n_varying} of {dup_groups.ngroups} product/size groups have rows with genuinely different calories/price/sugar/caffeine values')

menu[(menu['product_name'] == 'Caffè Latte') & (menu['size'] == 'Grande')]

24 of 106 product/size groups have rows with genuinely different calories/price/sugar/caffeine values


,product_name,category,size,price_usd,calories,sugar_g,caffeine_mg,seasonal_flag
5,Caffè Latte,Classic Espresso Drinks,Grande,4.21,100,9,75.0,0
6,Caffè Latte,Classic Espresso Drinks,Grande,4.49,70,4,75.0,0
8,Caffè Latte,Classic Espresso Drinks,Grande,4.18,150,14,75.0,0
9,Caffè Latte,Classic Espresso Drinks,Grande,4.33,110,6,75.0,0
10,Caffè Latte,Classic Espresso Drinks,Grande,4.18,130,18,150.0,0
11,Caffè Latte,Classic Espresso Drinks,Grande,4.18,190,17,150.0,0
12,Caffè Latte,Classic Espresso Drinks,Grande,4.29,150,8,150.0,0
14,Caffè Latte,Classic Espresso Drinks,Grande,3.99,240,22,150.0,0
15,Caffè Latte,Classic Espresso Drinks,Grande,4.17,190,11,150.0,0


Confirmed — this is a real data quality problem, not just harmless duplication. A Grande Caffè Latte alone shows up with prices from \$3.99 to \$4.49 and calories from 70 to 240. **Decision:** Menu will be deduplicated to one row per `(product_name, size)` in `03_data_cleaning.ipynb` before it's used as a dimension table. Since Transaction already carries its own `calories` / `sugar_g` / `caffeine_mg` / `base_price` at the row level (the synthetic generator snapshotted them at sale time), the only field the master dataset actually needs to pull from the cleaned Menu table is `seasonal_flag`.

## Grain mismatch: Macro is monthly, Transaction is daily

`Macro` has one row per calendar month; `Transaction` has one row per sale. Joining them means first truncating `Transaction.date` down to a month. Once that's done, does every transaction month have a matching macro row?

In [9]:
transactions['month'] = pd.to_datetime(transactions['date']).dt.to_period('M').astype(str)
macro['month']        = pd.to_datetime(macro['date']).dt.to_period('M').astype(str)

missing_months = set(transactions['month']) - set(macro['month'])
print('transaction months with no macro coverage:', missing_months)
print('affected transactions:', transactions['month'].isin(missing_months).sum())

transaction months with no macro coverage: {'2026-03'}
affected transactions: 122


Macro data ends at `2026-02-01`, but Transaction has rows in `2026-03`. Those rows will get a `NULL` `cpi` / `avg_hourly_earnings` / `real_wage_index` after the join — not a bug, just the fact table running ahead of the slowest-updating dimension. How to handle it (leave NULL, forward-fill the last known CPI, or drop those rows) is a cleaning decision, tracked in `03_data_cleaning.ipynb`.

## Findings summary

| Relationship | Status | Action |
|---|---|---|
| `Transaction.(product_name, size)` → `Menu.(product_name, size)` | Keys match, but Menu has duplicate/conflicting rows per key | Dedupe Menu in Phase 3 |
| `Transaction.category` → `Menu.category` | Clean, 1:1 | None needed |
| `Transaction.city` → `Weather.city` | Clean, 1:1 | None needed |
| `Transaction.(date, city)` → `Weather.(date, city)` | Clean, dates already overlap in range | None needed |
| `Transaction.month` → `Macro.month` | Clean except `2026-03` (122 rows, no macro coverage) | Decide null-handling strategy in Phase 3 |

Separately, the audit notebook's `.isnull().sum()` already flagged nulls worth carrying into Phase 3: `transactions.cpi` / `transactions.total_price` (4,324 each), `transactions.caffeine_mg` (8,186), `menu.caffeine_mg` (23), and `macro.cpi` / `macro.real_wage_index` (1 each). Those aren't relationship problems — they're single-column cleaning problems — but they matter here because they determine how much of the master dataset will actually have complete rows.